In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import os, csv, json, random, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter, defaultdict
 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
 
FEATURES_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2"
ZIP1 = os.path.join("/kaggle/input/datasets/ptrnghieu/hi-ef-dataset",
                     "Hi-EF-20260829T071606Z-1-001", "Hi-EF")
 
annotations = {}
with open(os.path.join(ZIP1, "annotation.csv"), 'r') as f:
    for row in csv.reader(f):
        if len(row) >= 2:
            cid = row[0].strip()
            annotations[cid] = {
                'emotion': row[7].strip() if len(row)>7 and row[7].strip() else None,
                'polarity': row[5].strip() if len(row)>5 and row[5].strip() else None,
                'intensity': row[6].strip() if len(row)>6 and row[6].strip() else None,
                'scene': row[4].strip() if len(row)>4 and row[4].strip() else None,
            }
 
samples = []
with open(os.path.join(ZIP1, "sample.csv"), 'r') as f:
    reader = csv.reader(f); next(reader)
    for row in reader: samples.append(row)
 
emotion_map = {'angry':0,'disgust':1,'fear':2,'happy':3,'neutral':4,'sad':5,'surprise':6}
emo_names = ['angry','disgust','fear','happy','neutral','sad','surprise']
 
mcis_index = []
for s in samples:
    clips = [s[i].strip() for i in range(1, 5)]
    entry = {'sample_id': s[0].strip(), 'clip_ids': clips,
             'feature_files': [c.replace('/','_')+'.pt' for c in clips],
             'show': clips[2].split('/')[0]}  # show number from clip3
    for ln, ci in [('clip3',2),('clip4',3)]:
        cid = clips[ci]
        if cid in annotations and annotations[cid].get('emotion'):
            ann = annotations[cid]
            entry[f'{ln}_emotion'] = emotion_map.get(ann['emotion'],-1)
            entry[f'{ln}_scene'] = ann.get('scene','')
        else:
            entry[f'{ln}_emotion'] = -1; entry[f'{ln}_scene'] = ''
    mcis_index.append(entry)
 
def split_mcis(idx_list, seed=42):
    rng = random.Random(seed)
    c4g = {}
    for i, e in enumerate(idx_list):
        c4 = e['clip_ids'][3]; c4g.setdefault(c4, []).append(i)
    keys = list(c4g.keys()); rng.shuffle(keys)
    n = len(keys); nt = int(n*0.7); nv = int(n*0.15)
    return ([i for g in keys[:nt] for i in c4g[g]],
            [i for g in keys[nt:nt+nv] for i in c4g[g]],
            [i for g in keys[nt+nv:] for i in c4g[g]])
 
train_idx, val_idx, test_idx = split_mcis(mcis_index)
print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")


Device: cuda
Train: 1980, Val: 424, Test: 426


In [3]:
BATCH_SIZE = 32; N_EPOCHS = 30; SEED = 42
 
class TemporalTransformer(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1,16,d)*0.02)
        el = nn.TransformerEncoderLayer(d,8,d*4,0.1,batch_first=True,norm_first=True)
        self.t = nn.TransformerEncoder(el, num_layers=2)
    def forward(self, x): return self.t(x + self.pos[:,:x.size(1),:])
 
class CrossAttentionFusion(nn.Module):
    def __init__(self, d=512, n_layers=1):
        super().__init__()
        self.layers = nn.ModuleList([nn.MultiheadAttention(d,8,dropout=0.1,batch_first=True) for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d) for _ in range(n_layers)])
    def forward(self, q, kv):
        x = q
        for a, n in zip(self.layers, self.norms):
            o, _ = a(x, kv, kv); x = n(x + o)
        return x
 
class ClipEncoder(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.ft = TemporalTransformer(d); self.ot = TemporalTransformer(d)
        self.tf = CrossAttentionFusion(d); self.ap = nn.Linear(527,d); self.mf = CrossAttentionFusion(d)
    def forward(self, face, ori, text, audio):
        f = self.ft(face).mean(1,keepdim=True); o = self.ot(ori).mean(1,keepdim=True)
        v = self.tf(f, torch.cat([f,o],1))
        a = self.ap(F.normalize(audio,dim=-1)).unsqueeze(1); t = text.unsqueeze(1)
        return self.mf(v, torch.cat([v,t,a],1)).squeeze(1)
 
def make_head(d=512, n=7):
    return nn.Sequential(nn.LayerNorm(d),nn.Dropout(0.3),nn.Linear(d,d//2),nn.GELU(),nn.Dropout(0.2),nn.Linear(d//2,n))
 
class M3_Full(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.enc = ClipEncoder(d)
        self.lstm = nn.LSTM(d,d,num_layers=3,batch_first=False,dropout=0.1)
        self.pos = nn.Parameter(torch.randn(1,3,d)*0.02)
        el = nn.TransformerEncoderLayer(d,8,d*4,0.1,batch_first=True,norm_first=True)
        self.trans = nn.TransformerEncoder(el, num_layers=2)
        self.head = make_head(d)
    def forward(self, batch):
        c1 = self.enc(batch['clip1_face'],batch['clip1_ori'],batch['clip1_text'],batch['clip1_audio'])
        c2 = self.enc(batch['clip2_face'],batch['clip2_ori'],batch['clip2_text'],batch['clip2_audio'])
        c3 = self.enc(batch['clip3_face'],batch['clip3_ori'],batch['clip3_text'],batch['clip3_audio'])
        seq = torch.stack([c1,c2,c3],0); out,_ = self.lstm(seq)
        return self.head(self.trans(out.permute(1,0,2)+self.pos).mean(1))
 
def compute_metrics(preds, labels):
    p, l = np.array(preds), np.array(labels)
    war = (p==l).sum()/len(l)*100
    rs = [((p[l==c]==c).sum()/(l==c).sum()*100) for c in range(7) if (l==c).sum()>0]
    return war, np.mean(rs) if rs else 0.0
 
def train_eval(model, trl, val, tst, n_epochs=30, name="m"):
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
    best_uar=0
    for ep in range(1,n_epochs+1):
        model.train()
        for b in trl:
            bg={k:v.to(DEVICE) for k,v in b.items()}
            loss=F.cross_entropy(model(bg),bg['target'])
            opt.zero_grad();loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step()
        model.eval();vp,vl=[],[];vl_loss=0
        with torch.no_grad():
            for b in val:
                bg={k:v.to(DEVICE) for k,v in b.items()};lo=model(bg)
                vl_loss+=F.cross_entropy(lo,bg['target']).item()*lo.size(0)
                vp.extend(lo.argmax(-1).cpu().numpy());vl.extend(b['target'].numpy())
        sch.step(vl_loss/len(val.dataset));_,vu=compute_metrics(vp,vl)
        if vu>best_uar: best_uar=vu; torch.save(model.state_dict(),f'/kaggle/working/{name}.pt')
    model.load_state_dict(torch.load(f'/kaggle/working/{name}.pt',map_location=DEVICE,weights_only=True))
    model.eval();tp,tl=[],[]
    with torch.no_grad():
        for b in tst:
            bg={k:v.to(DEVICE) for k,v in b.items()}
            tp.extend(model(bg).argmax(-1).cpu().numpy());tl.extend(b['target'].numpy())
    w,u=compute_metrics(tp,tl)
    return {'war':w,'uar':u,'preds':np.array(tp),'labels':np.array(tl)}

In [4]:
class FlexDataset(Dataset):
    """
    clip3_mode controls what goes in clip III position:
      'true'         — real clip III (default)
      'shuffle'      — random clip III from dataset
      'zero'         — zero tensors
      'noise'        — random Gaussian noise (fresh each time)
      'learned'      — not here (handled in model)
      'same_show'    — random clip III from same TV show
      'same_scene'   — random clip III from same scene type
      'same_emotion' — random clip III with same emotion label
    
    For label models:
      label_mode: 'true' or 'shuffle'
    """
    def __init__(self, mcis_index, features_dir, indices, clip3_mode='true', seed=42):
        self.mcis_index = mcis_index
        self.features_dir = features_dir
        self.indices = indices
        self.clip3_mode = clip3_mode
        self.rng = random.Random(seed)
        
        # Build lookup tables for structured shuffles
        self.by_show = defaultdict(list)
        self.by_scene = defaultdict(list)
        self.by_emotion = defaultdict(list)
        self.all_clip3_files = []
        
        for idx in indices:
            e = mcis_index[idx]
            f3 = e['feature_files'][2]
            self.all_clip3_files.append(f3)
            self.by_show[e['show']].append(f3)
            self.by_scene[e['clip3_scene']].append(f3)
            self.by_emotion[e['clip3_emotion']].append(f3)
    
    def __len__(self): return len(self.indices)
    
    def _load(self, ff):
        d = torch.load(os.path.join(self.features_dir, ff), map_location='cpu', weights_only=False)
        return d['face_features'], d['ori_features'], d['text_feature'], d.get('audio_feature', torch.zeros(527))
    
    def _get_clip3(self, idx):
        e = self.mcis_index[self.indices[idx]]
        mode = self.clip3_mode
        
        if mode == 'true':
            return self._load(e['feature_files'][2]), e['clip3_emotion']
        
        elif mode == 'zero':
            return (torch.zeros(16,512), torch.zeros(16,512), torch.zeros(512), torch.zeros(527)), e['clip3_emotion']
        
        elif mode == 'noise':
            return (torch.randn(16,512), torch.randn(16,512), torch.randn(512), torch.randn(527)), e['clip3_emotion']
        
        elif mode == 'shuffle':
            f3 = self.rng.choice(self.all_clip3_files)
            feats = self._load(f3)
            # Get emotion of the shuffled clip (unknown, use -1)
            return feats, -1
        
        elif mode == 'same_show':
            pool = self.by_show.get(e['show'], self.all_clip3_files)
            f3 = self.rng.choice(pool)
            return self._load(f3), -1
        
        elif mode == 'same_scene':
            pool = self.by_scene.get(e['clip3_scene'], self.all_clip3_files)
            f3 = self.rng.choice(pool)
            return self._load(f3), -1
        
        elif mode == 'same_emotion':
            pool = self.by_emotion.get(e['clip3_emotion'], self.all_clip3_files)
            f3 = self.rng.choice(pool)
            return self._load(f3), e['clip3_emotion']
        
        else:
            return self._load(e['feature_files'][2]), e['clip3_emotion']
    
    def __getitem__(self, idx):
        e = self.mcis_index[self.indices[idx]]
        c1f,c1o,c1t,c1a = self._load(e['feature_files'][0])
        c2f,c2o,c2t,c2a = self._load(e['feature_files'][1])
        (c3f,c3o,c3t,c3a), c3_emo = self._get_clip3(idx)
        
        return {
            'clip1_face':c1f,'clip1_ori':c1o,'clip1_text':c1t,'clip1_audio':c1a,
            'clip2_face':c2f,'clip2_ori':c2o,'clip2_text':c2t,'clip2_audio':c2a,
            'clip3_face':c3f,'clip3_ori':c3o,'clip3_text':c3t,'clip3_audio':c3a,
            'target':e['clip4_emotion'], 'clip3_emotion':c3_emo,
        }
 
def collate_fn(batch):
    r = {}
    for k in [f'clip{c}_{m}' for c in [1,2,3] for m in ['face','ori','text','audio']]:
        r[k] = torch.stack([b[k] for b in batch])
    for k in ['target','clip3_emotion']:
        r[k] = torch.tensor([b[k] for b in batch], dtype=torch.long)
    return r
 
def make_loaders(clip3_mode='true', seed=SEED):
    trl = DataLoader(FlexDataset(mcis_index,FEATURES_DIR,train_idx,clip3_mode,seed),
                     batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate_fn,num_workers=2,pin_memory=True)
    val = DataLoader(FlexDataset(mcis_index,FEATURES_DIR,val_idx,clip3_mode,seed+1),
                     batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_fn,num_workers=2,pin_memory=True)
    tst = DataLoader(FlexDataset(mcis_index,FEATURES_DIR,test_idx,clip3_mode,seed+2),
                     batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_fn,num_workers=2,pin_memory=True)
    return trl, val, tst
 
# Standard test loader (always true A, for cross-evaluation)
tst_true = DataLoader(FlexDataset(mcis_index,FEATURES_DIR,test_idx,'true'),
                       batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_fn,num_workers=2,pin_memory=True)
 
print("Setup complete ✓")

Setup complete ✓


In [5]:
print("=" * 70)
print("EXP 1: Parameter-Matched Null Controls (all 27.7M params)")
print("=" * 70)
 
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
 
null_configs = [
    ('true',     'M3(true A)'),
    ('zero',     'M3(zeros)'),
    ('noise',    'M3(noise)'),
    ('shuffle',  'M3(global shuffle)'),
]
 
null_results = {}
for mode, label in null_configs:
    print(f"\n  {label}...", end=" ", flush=True)
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    trl, val, tst = make_loaders(mode)
    m = M3_Full().to(DEVICE)
    r = train_eval(m, trl, val, tst, N_EPOCHS, name=f"null_{mode}")
    null_results[mode] = r
    print(f"WAR={r['war']:.1f}%, UAR={r['uar']:.1f}%")
 
# Also test each model on TRUE test data
print("\n  Cross-eval on true test data:")
for mode, label in null_configs:
    m = M3_Full().to(DEVICE)
    m.load_state_dict(torch.load(f'/kaggle/working/null_{mode}.pt',map_location=DEVICE,weights_only=True))
    m.eval(); tp,tl=[],[]
    with torch.no_grad():
        for b in tst_true:
            bg={k:v.to(DEVICE) for k,v in b.items()}
            tp.extend(m(bg).argmax(-1).cpu().numpy());tl.extend(b['target'].numpy())
    w,u = compute_metrics(tp,tl)
    null_results[f'{mode}_on_true'] = {'war':w,'uar':u}
    print(f"    {label} tested on true data: WAR={w:.1f}%, UAR={u:.1f}%")
 
print(f"\n--- Summary ---")
print(f"  {'Config':<25} | {'Train/Test UAR':>14} | {'On True Test':>12}")
print(f"  {'-'*55}")
for mode, label in null_configs:
    u1 = null_results[mode]['uar']
    u2 = null_results.get(f'{mode}_on_true', {}).get('uar', 0)
    print(f"  {label:<25} | {u1:>13.1f}% | {u2:>11.1f}%")

EXP 1: Parameter-Matched Null Controls (all 27.7M params)

  M3(true A)... 

/tmp/ipykernel_58/1835209706.py:8: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.t = nn.TransformerEncoder(el, num_layers=2)
/tmp/ipykernel_58/1835209706.py:43: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.trans = nn.TransformerEncoder(el, num_layers=2)


WAR=37.1%, UAR=25.7%

  M3(zeros)... 

/tmp/ipykernel_58/1835209706.py:8: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.t = nn.TransformerEncoder(el, num_layers=2)
/tmp/ipykernel_58/1835209706.py:43: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.trans = nn.TransformerEncoder(el, num_layers=2)


WAR=38.3%, UAR=25.2%

  M3(noise)... WAR=34.7%, UAR=23.9%

  M3(global shuffle)... WAR=37.6%, UAR=26.0%

  Cross-eval on true test data:
    M3(true A) tested on true data: WAR=37.1%, UAR=25.7%
    M3(zeros) tested on true data: WAR=39.4%, UAR=26.6%
    M3(noise) tested on true data: WAR=35.2%, UAR=24.3%
    M3(global shuffle) tested on true data: WAR=37.3%, UAR=25.8%

--- Summary ---
  Config                    | Train/Test UAR | On True Test
  -------------------------------------------------------
  M3(true A)                |          25.7% |        25.7%
  M3(zeros)                 |          25.2% |        26.6%
  M3(noise)                 |          23.9% |        24.3%
  M3(global shuffle)        |          26.0% |        25.8%


In [6]:
print("\n" + "=" * 70)
print("EXP 2: Hierarchical Shuffle (test-time intervention on trained M3)")
print("=" * 70)
 
# Load best true-A model
m3_true = M3_Full().to(DEVICE)
m3_true.load_state_dict(torch.load('/kaggle/working/null_true.pt',map_location=DEVICE,weights_only=True))
m3_true.eval()
 
shuffle_modes = [
    ('true',         'True A (baseline)'),
    ('same_emotion', 'Same emotion, diff person'),
    ('same_scene',   'Same scene type'),
    ('same_show',    'Same TV show'),
    ('shuffle',      'Global random'),
    ('zero',         'Zero vector'),
    ('noise',        'Random noise'),
]
 
hier_results = {}
for mode, label in shuffle_modes:
    tst_mode = DataLoader(FlexDataset(mcis_index,FEATURES_DIR,test_idx,mode,seed=42),
                           batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_fn,num_workers=2,pin_memory=True)
    tp,tl=[],[]
    with torch.no_grad():
        for b in tst_mode:
            bg={k:v.to(DEVICE) for k,v in b.items()}
            tp.extend(m3_true(bg).argmax(-1).cpu().numpy());tl.extend(b['target'].numpy())
    w,u = compute_metrics(tp,tl)
    hier_results[mode] = {'war':w,'uar':u}
    print(f"  {label:<30}: WAR={w:.1f}%, UAR={u:.1f}%")



EXP 2: Hierarchical Shuffle (test-time intervention on trained M3)


/tmp/ipykernel_58/1835209706.py:8: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.t = nn.TransformerEncoder(el, num_layers=2)
/tmp/ipykernel_58/1835209706.py:43: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.trans = nn.TransformerEncoder(el, num_layers=2)


  True A (baseline)             : WAR=37.1%, UAR=25.7%
  Same emotion, diff person     : WAR=35.7%, UAR=25.8%
  Same scene type               : WAR=36.9%, UAR=26.7%
  Same TV show                  : WAR=35.2%, UAR=24.8%
  Global random                 : WAR=32.4%, UAR=23.5%
  Zero vector                   : WAR=33.8%, UAR=23.9%
  Random noise                  : WAR=32.4%, UAR=24.8%


In [7]:
print("\n" + "=" * 70)
print("EXP 3: Train × Test Intervention Matrix")
print("=" * 70)
 
# We need 4 combos:
# train_true  × test_true    → already have (null_results['true'])
# train_true  × test_shuffle → test existing true model on shuffled data
# train_shuf  × test_shuf    → already have (null_results['shuffle'])
# train_shuf  × test_true    → test shuffled model on true data (null_results['shuffle_on_true'])
 
# train_true × test_shuffle
tst_shuf = DataLoader(FlexDataset(mcis_index,FEATURES_DIR,test_idx,'shuffle',seed=42),
                       batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_fn,num_workers=2,pin_memory=True)
m3_true.eval(); tp,tl=[],[]
with torch.no_grad():
    for b in tst_shuf:
        bg={k:v.to(DEVICE) for k,v in b.items()}
        tp.extend(m3_true(bg).argmax(-1).cpu().numpy());tl.extend(b['target'].numpy())
w_ts, u_ts = compute_metrics(tp,tl)
 
print(f"\n  {'':>20} | {'Test True':>10} | {'Test Shuffle':>12}")
print(f"  {'-'*48}")
 
r_tt = null_results['true']
r_ss = null_results['shuffle']
r_st = null_results['shuffle_on_true']
 
print(f"  {'Train True':<20} | {r_tt['uar']:>9.1f}% | {u_ts:>11.1f}%")
print(f"  {'Train Shuffle':<20} | {r_st['uar']:>9.1f}% | {r_ss['uar']:>11.1f}%")
 
drop_test_shuffle = r_tt['uar'] - u_ts
print(f"\n  Train(true) → Test(shuffle) drop: {drop_test_shuffle:+.1f} pts")
if abs(drop_test_shuffle) < 2:
    print(f"  ⚠ Model ignores A at test time — swapping A barely changes predictions")
else:
    print(f"  Model uses A at test time — swapping A changes predictions")



EXP 3: Train × Test Intervention Matrix

                       |  Test True | Test Shuffle
  ------------------------------------------------
  Train True           |      25.7% |        23.5%
  Train Shuffle        |      25.8% |        26.0%

  Train(true) → Test(shuffle) drop: +2.2 pts
  Model uses A at test time — swapping A changes predictions


In [8]:
print("\n" + "=" * 70)
print("EXP 4: A-Label Intervention (E_A true vs E_A shuffle)")
print("=" * 70)
 
class M_Label(nn.Module):
    """P(B|C, E_A): context + emotion label embedding."""
    def __init__(self, d=512):
        super().__init__()
        self.enc = ClipEncoder(d)
        self.ctx_f = CrossAttentionFusion(d)
        self.emo_emb = nn.Embedding(7, d)
        self.combine = CrossAttentionFusion(d, n_layers=2)
        self.head = make_head(d)
    def forward(self, batch):
        c1 = self.enc(batch['clip1_face'],batch['clip1_ori'],batch['clip1_text'],batch['clip1_audio'])
        c2 = self.enc(batch['clip2_face'],batch['clip2_ori'],batch['clip2_text'],batch['clip2_audio'])
        ctx = self.ctx_f(c1.unsqueeze(1), torch.stack([c1,c2],1)).squeeze(1)
        emo = self.emo_emb(batch['clip3_emotion'].clamp(min=0))
        combined = self.combine(emo.unsqueeze(1), torch.cat([emo.unsqueeze(1),ctx.unsqueeze(1)],1)).squeeze(1)
        return self.head(combined)
 
# Train with true labels
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
trl_t, val_t, tst_t = make_loaders('true')
 
print("  M_label (true E_A)...", end=" ", flush=True)
m_lt = M_Label().to(DEVICE)
r_lt = train_eval(m_lt, trl_t, val_t, tst_t, N_EPOCHS, name="m_label_true")
print(f"WAR={r_lt['war']:.1f}%, UAR={r_lt['uar']:.1f}%")
 
# Test trained model with SHUFFLED labels at test time
m_lt.load_state_dict(torch.load('/kaggle/working/m_label_true.pt',map_location=DEVICE,weights_only=True))
m_lt.eval()
 
# Create test set with shuffled emotion labels
class LabelShuffleDataset(Dataset):
    def __init__(self, base_dataset, seed=42):
        self.base = base_dataset
        self.rng = random.Random(seed)
        # Collect all clip3 emotions from training data
        all_emos = [mcis_index[i]['clip3_emotion'] for i in train_idx if mcis_index[i]['clip3_emotion']>=0]
        self.all_emos = all_emos
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        item = self.base[idx]
        item = dict(item)
        item['clip3_emotion'] = self.rng.choice(self.all_emos)
        return item
 
tst_label_shuf = DataLoader(LabelShuffleDataset(FlexDataset(mcis_index,FEATURES_DIR,test_idx,'true')),
                              batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_fn,num_workers=2,pin_memory=True)
 
tp,tl=[],[]
with torch.no_grad():
    for b in tst_label_shuf:
        bg={k:v.to(DEVICE) for k,v in b.items()}
        tp.extend(m_lt(bg).argmax(-1).cpu().numpy());tl.extend(b['target'].numpy())
w_ls, u_ls = compute_metrics(tp,tl)
 
print(f"  M_label (shuffled E_A at test): WAR={w_ls:.1f}%, UAR={u_ls:.1f}%")
 
label_drop = r_lt['uar'] - u_ls
print(f"\n  Drop from true→shuffle label: {label_drop:+.1f} pts")
if abs(label_drop) < 2:
    print(f"  ⚠ Model ignores emotion LABEL — shuffling barely changes output")
else:
    print(f"  ✓ Model uses emotion label — shuffling hurts by {label_drop:.1f} pts")



EXP 4: A-Label Intervention (E_A true vs E_A shuffle)
  M_label (true E_A)... 

/tmp/ipykernel_58/1835209706.py:8: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.t = nn.TransformerEncoder(el, num_layers=2)


WAR=38.5%, UAR=25.9%
  M_label (shuffled E_A at test): WAR=27.7%, UAR=18.3%

  Drop from true→shuffle label: +7.6 pts
  ✓ Model uses emotion label — shuffling hurts by 7.6 pts


In [11]:
print("\n" + "=" * 70)
print("EXP 5: Can context alone predict A's emotion?")
print("=" * 70)
 
class ContextProbe(nn.Module):
    """Predict A's emotion from clips I+II only."""
    def __init__(self, d=512):
        super().__init__()
        self.enc = ClipEncoder(d)
        self.ctx_f = CrossAttentionFusion(d)
        self.head = make_head(d)
    def forward(self, batch):
        c1 = self.enc(batch['clip1_face'],batch['clip1_ori'],batch['clip1_text'],batch['clip1_audio'])
        c2 = self.enc(batch['clip2_face'],batch['clip2_ori'],batch['clip2_text'],batch['clip2_audio'])
        ctx = self.ctx_f(c1.unsqueeze(1), torch.stack([c1,c2],1)).squeeze(1)
        return self.head(ctx)
 
# Need dataset that returns clip3_emotion as target (not clip4)
class ProbeDataset(Dataset):
    def __init__(self, mcis_index, features_dir, indices):
        self.mcis_index = mcis_index; self.features_dir = features_dir
        self.indices = [i for i in indices if mcis_index[i]['clip3_emotion'] >= 0]
    def __len__(self): return len(self.indices)
    def _load(self, ff):
        d = torch.load(os.path.join(self.features_dir, ff), map_location='cpu', weights_only=False)
        return d['face_features'], d['ori_features'], d['text_feature'], d.get('audio_feature', torch.zeros(527))
    def __getitem__(self, idx):
        e = self.mcis_index[self.indices[idx]]
        c1f,c1o,c1t,c1a = self._load(e['feature_files'][0])
        c2f,c2o,c2t,c2a = self._load(e['feature_files'][1])
        return {
            'clip1_face':c1f,'clip1_ori':c1o,'clip1_text':c1t,'clip1_audio':c1a,
            'clip2_face':c2f,'clip2_ori':c2o,'clip2_text':c2t,'clip2_audio':c2a,
            # Dummy clip3 (not used by probe, but collate expects it)
            'clip3_face':torch.zeros(16,512),'clip3_ori':torch.zeros(16,512),
            'clip3_text':torch.zeros(512),'clip3_audio':torch.zeros(527),
            'target': e['clip3_emotion'],  # predict A's emotion!
            'clip3_emotion': e['clip3_emotion'],
        }
 
trl_p = DataLoader(ProbeDataset(mcis_index,FEATURES_DIR,train_idx),batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate_fn,num_workers=2,pin_memory=True)
val_p = DataLoader(ProbeDataset(mcis_index,FEATURES_DIR,val_idx),batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_fn,num_workers=2,pin_memory=True)
tst_p = DataLoader(ProbeDataset(mcis_index,FEATURES_DIR,test_idx),batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_fn,num_workers=2,pin_memory=True)
 
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print("  Training context→E_A probe...", end=" ", flush=True)
probe = ContextProbe().to(DEVICE)
r_probe = train_eval(probe, trl_p, val_p, tst_p, N_EPOCHS, name="probe_ctx2ea")
print(f"WAR={r_probe['war']:.1f}%, UAR={r_probe['uar']:.1f}%")
 
# Majority baseline for A's emotion
train_ea = [mcis_index[i]['clip3_emotion'] for i in train_idx if mcis_index[i]['clip3_emotion']>=0]
maj_ea = Counter(train_ea).most_common(1)[0]
test_ea = [mcis_index[i]['clip3_emotion'] for i in test_idx if mcis_index[i]['clip3_emotion']>=0]
maj_preds = [maj_ea[0]] * len(test_ea)
maj_w, maj_u = compute_metrics(maj_preds, test_ea)
 
print(f"  Majority baseline for E_A: WAR={maj_w:.1f}%, UAR={maj_u:.1f}%")
print(f"  Probe improvement over majority: +{r_probe['uar']-maj_u:.1f} pts")
 
if r_probe['uar'] > maj_u + 5:
    print(f"\n  ✓ Context CAN predict A's emotion ({r_probe['uar']:.1f}% >> {maj_u:.1f}%)")
    print(f"  → This explains why clip III is redundant: context already encodes A's likely state")
else:
    print(f"\n  ~ Context weakly predicts A's emotion")
    print(f"  → A's redundancy comes from A being generally uninformative, not from context overlap")



EXP 5: Can context alone predict A's emotion?
  Training context→E_A probe... 

/tmp/ipykernel_58/1835209706.py:8: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.t = nn.TransformerEncoder(el, num_layers=2)


WAR=35.7%, UAR=26.9%
  Majority baseline for E_A: WAR=22.5%, UAR=14.3%
  Probe improvement over majority: +12.6 pts

  ✓ Context CAN predict A's emotion (26.9% >> 14.3%)
  → This explains why clip III is redundant: context already encodes A's likely state


In [12]:
print("\n" + "=" * 70)
print("GRAND SUMMARY: Does A Drive B's Emotion?")
print("=" * 70)
 
print(f"""
╔══════════════════════════════════════════════════════════════════════╗
║  EXP 1: NULL CONTROLS (same 27.7M params)                         ║
║    M3(true A):     {null_results['true']['uar']:>5.1f}%                                      ║
║    M3(zeros):      {null_results['zero']['uar']:>5.1f}%                                      ║
║    M3(noise):      {null_results['noise']['uar']:>5.1f}%                                      ║
║    M3(shuffle):    {null_results['shuffle']['uar']:>5.1f}%                                      ║
║    → Conclusion: {'A adds NO information beyond capacity' if null_results['true']['uar'] <= null_results['shuffle']['uar'] + 1 else 'A adds genuine information':<42}║
╠══════════════════════════════════════════════════════════════════════╣
║  EXP 2: HIERARCHICAL SHUFFLE (test-time on trained M3)            ║
║    True A:         {hier_results['true']['uar']:>5.1f}%                                      ║
║    Same emotion:   {hier_results['same_emotion']['uar']:>5.1f}%                                      ║
║    Same scene:     {hier_results['same_scene']['uar']:>5.1f}%                                      ║
║    Same show:      {hier_results['same_show']['uar']:>5.1f}%                                      ║
║    Global random:  {hier_results['shuffle']['uar']:>5.1f}%                                      ║
║    Zeros:          {hier_results['zero']['uar']:>5.1f}%                                      ║
╠══════════════════════════════════════════════════════════════════════╣
║  EXP 3: TRAIN × TEST MATRIX                                       ║
║                     Test True    Test Shuffle                      ║
║    Train True:      {r_tt['uar']:>5.1f}%       {u_ts:>5.1f}%                              ║
║    Train Shuffle:   {r_st['uar']:>5.1f}%       {r_ss['uar']:>5.1f}%                              ║
║    Drop(true→shuf): {drop_test_shuffle:>+5.1f} pts                                     ║
╠══════════════════════════════════════════════════════════════════════╣
║  EXP 4: LABEL INTERVENTION                                        ║
║    P(B|C, E_A true):    {r_lt['uar']:>5.1f}%                                      ║
║    P(B|C, E_A shuffle): {u_ls:>5.1f}%                                      ║
║    Drop: {label_drop:>+5.1f} pts                                                ║
╠══════════════════════════════════════════════════════════════════════╣
║  EXP 5: CONTEXT → E_A PROBE                                       ║
║    Can C predict E_A?   {r_probe['uar']:>5.1f}% (majority: {maj_u:.1f}%)                 ║
╚══════════════════════════════════════════════════════════════════════╝
""")
 
# Save all results
save = {
    'null_controls': {k: {'war':v['war'],'uar':v['uar']} for k,v in null_results.items()},
    'hierarchy': {k: {'war':v['war'],'uar':v['uar']} for k,v in hier_results.items()},
    'train_test_matrix': {
        'train_true_test_true': r_tt['uar'],
        'train_true_test_shuffle': u_ts,
        'train_shuffle_test_true': r_st['uar'],
        'train_shuffle_test_shuffle': r_ss['uar'],
    },
    'label_intervention': {
        'true_label': r_lt['uar'],
        'shuffled_label': u_ls,
        'drop': label_drop,
    },
    'context_probe': {
        'probe_uar': r_probe['uar'],
        'majority_uar': maj_u,
    },
}
with open('/kaggle/working/null_controls_results.json', 'w') as f:
    json.dump(save, f, indent=2)
print("Saved to /kaggle/working/null_controls_results.json")



GRAND SUMMARY: Does A Drive B's Emotion?

╔══════════════════════════════════════════════════════════════════════╗
║  EXP 1: NULL CONTROLS (same 27.7M params)                         ║
║    M3(true A):      25.7%                                      ║
║    M3(zeros):       25.2%                                      ║
║    M3(noise):       23.9%                                      ║
║    M3(shuffle):     26.0%                                      ║
║    → Conclusion: A adds NO information beyond capacity     ║
╠══════════════════════════════════════════════════════════════════════╣
║  EXP 2: HIERARCHICAL SHUFFLE (test-time on trained M3)            ║
║    True A:          25.7%                                      ║
║    Same emotion:    25.8%                                      ║
║    Same scene:      26.7%                                      ║
║    Same show:       24.8%                                      ║
║    Global random:   23.5%                                      ║
║    